# Scalability Analysis
Plots pipeline throughput and latency as data volume grows from 10% → 50% → 100%.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')

In [ ]:
df = pd.read_csv('/data/results/scalability.csv')
df['pct'] = (df['fraction'] * 100).astype(int)
df

## 1. Processing time vs data fraction (per phase)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

phases = ['ingest', 'er', 'graph_prep']
titles = ['Data Ingestion', 'Entity Resolution (LSH)', 'Graph Build Prep']
colors = ['#2980b9', '#e74c3c', '#27ae60']

for ax, phase, title, color in zip(axes, phases, titles, colors):
    sub = df[df['phase'] == phase]
    ax.plot(sub['pct'], sub['time_s'], 'o-', color=color, linewidth=2, markersize=10)
    for _, row in sub.iterrows():
        ax.annotate(f"{row['time_s']:.1f}s",
                    (row['pct'], row['time_s']),
                    textcoords='offset points', xytext=(5, 8), fontsize=10)
    ax.set_xlabel('Data fraction (%)', fontsize=12)
    ax.set_ylabel('Time (s)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.set_xticks([10, 50, 100])

plt.suptitle('Pipeline Scalability (10% → 50% → 100% data)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/data/results/scalability_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Throughput (records/s) vs fraction

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for phase, color, label in zip(phases, colors, titles):
    sub = df[df['phase'] == phase]
    ax.plot(sub['pct'], sub['throughput_rps'], 'o-', color=color, linewidth=2, markersize=9, label=label)

ax.set_xlabel('Data fraction (%)', fontsize=12)
ax.set_ylabel('Throughput (records/s)', fontsize=12)
ax.set_title('Throughput vs Data Volume', fontsize=13)
ax.set_xticks([10, 50, 100])
ax.legend()
plt.tight_layout()
plt.savefig('/data/results/scalability_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Skew benchmark — naive vs salted join

In [ ]:
skew = pd.read_csv('/data/results/spark_skew.csv')

summary = skew.groupby('method')['time_s'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(summary['method'], summary['mean'],
              yerr=summary['std'], capsize=6,
              color=['#e74c3c', '#2ecc71'])
ax.set_ylabel('Join time (s)', fontsize=12)
ax.set_title('Skew Handling: Naive vs Salted Join (3 runs)', fontsize=13)
ax.bar_label(bars, fmt='%.2fs', fontsize=11, padding=4)
plt.tight_layout()
plt.savefig('/data/results/skew_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

naive_mean = summary.loc[summary['method'] == 'naive', 'mean'].values[0]
salted_mean = summary.loc[summary['method'].str.startswith('salted'), 'mean'].values[0]
print(f'Speedup: {naive_mean/salted_mean:.2f}x')

## 4. Partition strategy comparison

In [ ]:
part = pd.read_csv('/data/results/spark_partition.csv')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(part['strategy'], part['time_s'],
              color=sns.color_palette('Set2', len(part)))
ax.set_ylabel('Time (s)', fontsize=12)
ax.set_title('Partition Strategy: Join Time Comparison', fontsize=13)
ax.bar_label(bars, fmt='%.2fs', fontsize=10, padding=3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/data/results/partition_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Neo4j batch size benchmark

In [ ]:
neo = pd.read_csv('/data/results/graph_load_benchmark.csv')

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(neo['batch_size'], neo['time_s'], 'b-o', linewidth=2, markersize=8, label='Time (s)')
ax2.plot(neo['batch_size'], neo['records_per_s'], 'r--s', linewidth=2, markersize=8, label='Throughput (rec/s)')

ax1.set_xlabel('Batch size', fontsize=12)
ax1.set_ylabel('Time (s)', color='blue', fontsize=12)
ax2.set_ylabel('Records/s', color='red', fontsize=12)
ax1.set_title('Neo4j Write: Batch Size vs Performance', fontsize=13)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.savefig('/data/results/neo4j_batch.png', dpi=150, bbox_inches='tight')
plt.show()